# Jour 2 — Pandas, Fichiers et SQL · Notebook Formateur
## Formation Beobank · SAS → Python · Orsys

**Usage :** notebook du formateur — chaque ligne commentée, code à projeter.
**Données :** `../data/` — CTR.csv, TIE.csv, TIE_ADR.csv, TIE_X_CTR.csv, TXN_X_CTR.csv

📎 **Comparaisons SAS ↔ Python détaillées :** voir `../sas_vs_python/SAS_vs_Python.ipynb`.

⚠️ **Note données :** `TXN_X_CTR.csv` ne contient pas de montant ni de date de mouvement exploitable (colonnes absentes du fichier source). Les colonnes `MNT_MVT` et `DAT_MVT` utilisées à partir d'ici sont **simulées** (graine fixe, reproductibles) — voir la cellule Setup.

## Setup — Imports et chargement des 5 tables
### Cellule à exécuter en premier

In [1]:
# ── Imports — bibliothèques nécessaires ───────────────────────────────────
import pandas as pd          # Pandas = la bibliothèque de traitement de données
import numpy as np           # NumPy = calcul numérique (np.where, np.select...)
import sqlite3               # SQLite = base de données légère en mémoire
from pathlib import Path     # pathlib = gestion des chemins de fichiers

# ── Chemin des données ────────────────────────────────────────────────────────
# Path("../data") remonte d'un dossier depuis le notebook, puis entre dans data/
# L'opérateur / concatène les chemins proprement (Windows et Linux)
DATA = Path("../data")

# Paramètres communs à tous les fichiers Beobank
# sep=";"        → séparateur point-virgule
# na_values="."  → les "." (valeurs manquantes) deviennent NaN
# encoding="utf-8" → encodage pour les caractères accentués
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")

# ── Chargement des 5 tables ───────────────────────────────────────────────────
ctr     = pd.read_csv(DATA / "CTR.csv",       **PARAMS)   # 200 contrats
tie     = pd.read_csv(DATA / "TIE.csv",       **PARAMS)   # 100 clients
tie_adr = pd.read_csv(DATA / "TIE_ADR.csv",   **PARAMS)   # 100 adresses
txc     = pd.read_csv(DATA / "TIE_X_CTR.csv", **PARAMS)   # 200 liens
txn     = pd.read_csv(DATA / "TXN_X_CTR.csv", **PARAMS)   # 1260 transactions

# ── Données simulées pour l'exercice ──────────────────────────────────────
# TXN_X_CTR.csv ne fournit ni montant ni date de mouvement exploitable
# (uniquement des libellés). On simule ces deux colonnes pour les besoins
# pédagogiques : montant ~ loi normale, date = date de création du mouvement.
rng = np.random.default_rng(42)                          # graine fixe → résultats reproductibles
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_CRE_MVT_CPB"])   # date de mouvement simulée
txn["MNT_MVT"] = rng.normal(250, 400, size=len(txn)).round(2)  # montant simulé (moyenne 250€, écart-type 400€)

# Résumé du chargement
for nom, df in [("CTR",ctr),("TIE",tie),("TIE_ADR",tie_adr),
                ("TIE_X_CTR",txc),("TXN_X_CTR",txn)]:
    print(f"  {nom:12s} : {df.shape[0]:5d} lignes, {df.shape[1]:2d} colonnes")

  CTR          :   200 lignes, 11 colonnes
  TIE          :   100 lignes, 11 colonnes
  TIE_ADR      :   100 lignes, 20 colonnes
  TIE_X_CTR    :   200 lignes,  6 colonnes
  TXN_X_CTR    :  1260 lignes, 12 colonnes


## Module 1 — Explorer un DataFrame
### 1.1 head, tail, shape, info

In [ ]:
# ── Exploration de base ────────────────────────────────────────────────────
# Python :
print("=== 5 premières lignes (PROC PRINT OBS=5) ===")
print(ctr.head(5))           # 5 premières lignes

print("\n=== Types des colonnes (PROC CONTENTS) ===")
print(ctr.dtypes)            # type de chaque colonne

print("\n=== Valeurs manquantes par colonne ===")
print(ctr.isna().sum())      # nombre de NaN par colonne

# shape = (nb_lignes, nb_colonnes)
print(f"\nDimensions : {ctr.shape[0]} lignes, {ctr.shape[1]} colonnes")

=== 5 premières lignes (PROC PRINT OBS=5) ===
        IDT_AC  REF_CTR_INN DAT_OUV_CTR  COD_ECV_CTR DAT_ECV_CTR DAT_CLO_CTR  \
0  65500004701  29862201102  2024-05-29            6  2025-12-10  2025-12-10   
1  65500006391  29912218433  2024-08-07            6  2025-02-05  2025-02-05   
2  65500007774  29922113324  2022-01-12            4  2022-01-12         NaN   
3  65500008787  29872222935  2025-01-07            4  2025-01-07         NaN   
4  65500014230  29862237513  2025-01-18            6  2025-02-15  2025-02-15   

  COD_DEV  SLD_CTR DAT_MAJ_SLD  SLD_DSP  MNT_INI  
0     EUR      NaN         NaN      NaN      NaN  
1     EUR      NaN         NaN      NaN      NaN  
2     EUR      NaN         NaN      NaN      NaN  
3     EUR      NaN  2025-10-31      NaN      NaN  
4     EUR      NaN         NaN      NaN      NaN  

=== Types des colonnes (PROC CONTENTS) ===
IDT_AC           int64
REF_CTR_INN      int64
DAT_OUV_CTR        str
COD_ECV_CTR      int64
DAT_ECV_CTR        str
DAT_CLO_

: 

In [ ]:
# ── describe et value_counts — statistiques ────────────────────────────────
print("=== Statistiques descriptives (PROC MEANS) ===")
print(ctr[["SLD_CTR", "MNT_INI", "SLD_DSP"]].describe().round(2))

print("\n=== Distribution des statuts (PROC FREQ) ===")
freq = ctr["COD_ECV_CTR"].value_counts()
print(freq)

# Avec pourcentages
print("\n=== Pourcentages ===")
pct = ctr["COD_ECV_CTR"].value_counts(normalize=True) * 100
print(pct.round(1))

### ✏️ Exercice 1 — Explorer TIE (clients)

In [ ]:
# ── EXERCICE 1 ─────────────────────────────────────────────────────────────
# Sur la table TIE (clients), afficher :
# 1. head(3) + shape
# 2. value_counts sur COD_TYP_TIE (1=PP, 2=PM)
# 3. value_counts sur COD_LNG_CTR (FR/NL) avec normalize=True
# 4. isna().sum() — colonnes avec valeurs manquantes

print("=== TIE — clients ===")
print(tie.head(3))
print(f"\nDimensions : {tie.shape}")

# TODO : les 3 autres analyses

### ✅ Correction Exercice 1

In [ ]:
# ── CORRECTION EXERCICE 1 ─────────────────────────────────────────────────
print("=== Répartition PP vs PM ===")
print(tie["COD_TYP_TIE"].value_counts())     # 1=PP, 2=PM

print("\n=== Répartition FR vs NL (%) ===")
print((tie["COD_LNG_CTR"].value_counts(normalize=True) * 100).round(1))

print("\n=== Valeurs manquantes ===")
manquants = tie.isna().sum()
print(manquants[manquants > 0])             # afficher seulement colonnes avec NaN

## Module 2 — NumPy : tableaux et calcul vectorisé
### 2.1 Créer un tableau NumPy

In [ ]:
# ── NumPy — tableaux (arrays) et calcul vectorisé ─────────────────────────
# import numpy as np a déjà été fait dans le Setup — rappel ici
# Un array NumPy ressemble à une liste Python, mais permet des calculs
# GLOBAUX très rapides, sans écrire de boucle for.

soldes_liste = [15234.50, -500.0, 8900.0, 0.0, 32100.0]   # liste Python classique
soldes_array = np.array(soldes_liste)                       # conversion liste → array NumPy

print(type(soldes_liste))   # <class 'list'>
print(type(soldes_array))   # <class 'numpy.ndarray'>
print(soldes_array)

### 2.2 Opérations vectorisées — sans boucle for

In [ ]:
# ── Opérations vectorisées — appliquées à TOUS les éléments d'un coup ─────
# Avec une liste Python, il faudrait une boucle for pour calculer sur chaque élément.
# Avec un array NumPy, l'opération s'applique directement à tout le tableau :

interets = soldes_array * 0.035          # 3,5 % d'intérêt sur CHAQUE solde, sans boucle
print(interets.round(2))

solde_moins_frais = soldes_array - 5     # soustraire 5 EUR de frais à chaque solde
print(solde_moins_frais)

# Comparaison vectorisée : renvoie un array de True/False (un par élément)
masque_negatifs = soldes_array < 0
print(masque_negatifs)                    # [False  True False False False]

# Utiliser ce masque pour FILTRER le tableau (garder seulement les True)
print(soldes_array[masque_negatifs])      # [-500.]

### 2.3 Statistiques NumPy

In [ ]:
# ── Statistiques NumPy — résumer un tableau en quelques nombres ──────────
print(f"Moyenne    : {np.mean(soldes_array):.2f}")
print(f"Somme      : {np.sum(soldes_array):.2f}")
print(f"Minimum    : {np.min(soldes_array):.2f}")
print(f"Maximum    : {np.max(soldes_array):.2f}")
print(f"Écart-type : {np.std(soldes_array):.2f}")   # dispersion autour de la moyenne

### 2.4 Le lien avec Pandas

In [ ]:
# ── Une colonne Pandas EST un array NumPy en dessous ──────────────────────
# .to_numpy() extrait le tableau NumPy contenu dans une colonne (Series) Pandas
solde_array = ctr["SLD_CTR"].to_numpy()
print(type(solde_array))                              # <class 'numpy.ndarray'>
print(f"Solde moyen (NumPy) : {np.nanmean(solde_array):.2f}")   # nanmean ignore les valeurs manquantes

# C'est pour cette raison que "ctr['SLD_CTR'] * 2" fonctionne directement,
# sans boucle : Pandas utilise NumPy pour calculer sur toute la colonne d'un coup
print((ctr["SLD_CTR"] * 2).head(3))

### ✏️ Exercice 2 — Calculer des intérêts avec NumPy

In [ ]:
# ── EXERCICE ────────────────────────────────────────────────────────────────
soldes = [12000, -300, 8000, 45000, 150]

# TODO 1 : convertir soldes en array NumPy
# TODO 2 : calculer les intérêts à 2 % pour chaque solde (vectorisé, sans boucle)
# TODO 3 : afficher la moyenne, le min et le max des soldes (np.mean/min/max)
# TODO 4 : afficher uniquement les soldes négatifs avec un filtre booléen

### ✅ Correction Exercice 2

In [ ]:
# ── CORRECTION ─────────────────────────────────────────────────────────────
soldes = [12000, -300, 8000, 45000, 150]

# 1. Conversion en array
soldes_array = np.array(soldes)

# 2. Intérêts à 2 %, vectorisé
interets = soldes_array * 0.02
print("Intérêts :", interets.round(2))

# 3. Statistiques
print(f"Moyenne : {np.mean(soldes_array):.2f}")
print(f"Min     : {np.min(soldes_array)}")
print(f"Max     : {np.max(soldes_array)}")

# 4. Filtre booléen
print("Soldes négatifs :", soldes_array[soldes_array < 0])

## Module 3 — Sélection .loc et .iloc
### 3.1 .loc — par label

In [ ]:
# ── .loc — sélection par noms de colonnes et conditions ────────────────────

# Sélectionner une colonne entière (retourne une Series)
soldes = ctr.loc[:, "SLD_CTR"]         # toutes les lignes, colonne SLD_CTR
print(type(soldes))                    # <class 'pandas.core.series.Series'>

# Sélectionner plusieurs colonnes (retourne un DataFrame)
resume = ctr.loc[:, ["IDT_AC", "COD_ECV_CTR", "SLD_CTR", "COD_DEV"]]
print(resume.head(3))

# Sélectionner une ligne précise + une colonne
val = ctr.loc[0, "SLD_CTR"]           # ligne d'index 0, colonne SLD_CTR
print(f"Solde ligne 0 : {val}")

# Sélection avec condition (filtre — voir Module 3)
clotures = ctr.loc[ctr["COD_ECV_CTR"] == 4]   # code 4 = Clôturé (entier, pas texte)
print(f"Contrats clôturés : {len(clotures)}")

In [ ]:
# ── Renommer et supprimer des colonnes ────────────────────────────────────

# Supprimer des colonnes
ctr_light = ctr.drop(columns=["REF_CTR_INN"])
print(f"Colonnes après drop : {list(ctr_light.columns)}")

# Renommer des colonnes
ctr2 = ctr.rename(columns={"IDT_AC": "COMPTE_ID", "SLD_CTR": "SOLDE"})
print(f"Nouvelles colonnes : {list(ctr2.columns[:4])}")

# Sélectionner plusieurs colonnes simplement (raccourci)
cols = ["IDT_AC", "COD_ECV_CTR", "SLD_CTR"]
ctr_mini = ctr[cols]         # équivalent de ctr.loc[:, cols]
print(ctr_mini.head(2))

## Module 4 — Filtres booléens
### 4.1 Filtres de base

In [ ]:
# ── Filtres booléens — WHERE en Pandas ────────────────────────────────────
# Python : ctr[masque_booleen]
# COD_ECV_CTR est un entier dans le fichier réel (pas du texte) : on compare à 4, 6...

# Filtre simple : contrats clôturés (code 4 — présent dans notre extrait)
clotures = ctr[ctr["COD_ECV_CTR"] == 4]
print(f"Contrats clôturés : {len(clotures)}")

# ET logique : & (OBLIGATOIRE — ne pas utiliser 'and' entre colonnes Pandas)
# Chaque condition doit être entre PARENTHÈSES
clotures_eur = ctr[(ctr["COD_ECV_CTR"] == 4) & (ctr["COD_DEV"] == "EUR")]
print(f"Clôturés en EUR : {len(clotures_eur)}")

# OU logique : |
inactifs_ou = ctr[(ctr["COD_ECV_CTR"] == 4) | (ctr["COD_ECV_CTR"] == 6)]
print(f"Clôturés ou résiliés : {len(inactifs_ou)}")

# .isin() — tester l'appartenance à une liste de valeurs (raccourci du OU ci-dessus)
inactifs = ctr[ctr["COD_ECV_CTR"].isin([4, 6])]
print(f"Inactifs (isin) : {len(inactifs)}")

# .isna() / .notna() — valeurs manquantes
sans_solde = ctr[ctr["SLD_CTR"].isna()]
print(f"Sans solde : {len(sans_solde)}")

In [ ]:
# ── Tri — sort_values ────────────────────────────────────────────────────

# Tri croissant (défaut)
ctr_trie = ctr.sort_values("SLD_CTR")
print(ctr_trie[["IDT_AC", "SLD_CTR"]].head(5))

# Tri décroissant
ctr_trie_desc = ctr.sort_values("SLD_CTR", ascending=False)
print(ctr_trie_desc[["IDT_AC", "SLD_CTR"]].head(5))

# Tri sur plusieurs colonnes — d'abord COD_ECV_CTR, puis SLD_CTR décroissant
ctr_multi = ctr.sort_values(["COD_ECV_CTR", "SLD_CTR"], ascending=[True, False])
print(ctr_multi[["IDT_AC","COD_ECV_CTR","SLD_CTR"]].head(5))

### ✏️ Exercice 3 — Comptes à surveiller

In [ ]:
# ── EXERCICE 2 ─────────────────────────────────────────────────────────────
# 1. Extraire les contrats avec SLD_CTR < 0
# 2. Contrats clôturés (code 4) + EUR + SLD_CTR > 5000
# 3. Contrats clôturés ou résiliés (codes 4 ou 6) avec .isin()
# 4. Compter SLD_DSP manquant
# 5. Trier le résultat de (2) par SLD_CTR décroissant

# --- votre code ---

### ✅ Correction Exercice 3

In [ ]:
# ── CORRECTION EXERCICE 2 ─────────────────────────────────────────────────
# 1. Soldes négatifs
negatifs = ctr[ctr["SLD_CTR"] < 0]
print(f"1. Soldes négatifs : {len(negatifs)}")

# 2. Clôturés (code 4) + EUR + solde > 5000
top = ctr[(ctr["COD_ECV_CTR"]==4) & (ctr["COD_DEV"]=="EUR") & (ctr["SLD_CTR"]>5000)]
print(f"2. Clôturés EUR >5000 : {len(top)}")

# 3. Clôturés ou résiliés
inactifs = ctr[ctr["COD_ECV_CTR"].isin([4,6])]
print(f"3. Inactifs : {len(inactifs)}")

# 4. SLD_DSP manquant
print(f"4. SLD_DSP manquant : {ctr['SLD_DSP'].isna().sum()}")

# 5. Tri
top_trie = top.sort_values("SLD_CTR", ascending=False)
print("5. Top 5 comptes :")
print(top_trie[["IDT_AC","SLD_CTR","COD_ECV_CTR"]].head(5))

## Module 5 — Colonnes calculées
### 5.1 np.where et np.select

In [ ]:
# ── np.where — IF/ELSE en une ligne ──────────────────────────────────────
# Python :

# np.where(condition, valeur_si_vrai, valeur_si_faux)
ctr["FLAG_SOLDE"] = np.where(
    ctr["SLD_CTR"] > 0,    # condition : solde positif ?
    "POSITIF",             # si VRAI
    "NEGATIF_OU_NUL"       # si FAUX
)
print(ctr["FLAG_SOLDE"].value_counts())

In [ ]:
# ── np.select — IF/ELIF/ELSE multi-niveaux ────────────────────────────────

conditions = [
    ctr["SLD_CTR"] < 0,                          # niveau 1 : négatif
    ctr["SLD_CTR"].between(0, 5000),              # niveau 2 : faible
    ctr["SLD_CTR"].between(5000, 50000),          # niveau 3 : moyen
    ctr["SLD_CTR"] > 50000                        # niveau 4 : élevé
]
valeurs = ["Critique", "Faible", "Moyen", "Élevé"]

ctr["SEGMENT"] = np.select(
    conditions,
    valeurs,
    default="Non classé"    # si aucune condition (NaN par exemple)
)
print(ctr["SEGMENT"].value_counts())

In [ ]:
# ── .map(), pd.cut(), pd.to_datetime() ────────────────────────────────────
# .map() — appliquer un dictionnaire de correspondance
libelles = {1:"Ouvert",2:"En attente",3:"Suspendu",
            4:"Clôturé",5:"En résiliation",6:"Résilié"}
ctr["LIB_STATUT"] = ctr["COD_ECV_CTR"].map(libelles)

# pd.cut() — découper en tranches
ctr["TRANCHE"] = pd.cut(
    ctr["SLD_CTR"],
    bins=[0, 1000, 10000, 100000, float("inf")],
    labels=["<1k", "1k-10k", "10k-100k", ">100k"]
)

# pd.to_datetime() — parser les dates
ctr["DAT_OUV_CTR"] = pd.to_datetime(ctr["DAT_OUV_CTR"], format="mixed")
ctr["ANNEE_OUV"]   = ctr["DAT_OUV_CTR"].dt.year
ctr["MOIS_OUV"]    = ctr["DAT_OUV_CTR"].dt.month

print(ctr[["IDT_AC","LIB_STATUT","SEGMENT","TRANCHE","ANNEE_OUV"]].head(5))

### ✏️ Exercice 4 — Enrichir CTR

In [ ]:
# ── EXERCICE 3 ─────────────────────────────────────────────────────────────
# Sur ctr, créer :
# 1. ALERTE = "OUI" si SLD_CTR < 0 ou SLD_CTR < 100, sinon "NON" (np.where)
# 2. SEGMENT = 4 niveaux avec np.select (Critique/Faible/Moyen/Élevé)
# 3. LIB_STATUT avec .map()
# 4. Parser DAT_OUV_CTR → extraire ANNEE_OUV
# 5. value_counts() sur SEGMENT et ALERTE

# --- votre code ---

### ✅ Correction Exercice 4

In [ ]:
# ── CORRECTION EXERCICE 3 ─────────────────────────────────────────────────
# 1. ALERTE
ctr["ALERTE"] = np.where(
    (ctr["SLD_CTR"] < 0) | (ctr["SLD_CTR"] < 100),
    "OUI", "NON"
)

# 2. SEGMENT
cond = [ctr["SLD_CTR"]<0, ctr["SLD_CTR"]<5000,
        ctr["SLD_CTR"]<50000, ctr["SLD_CTR"]>=50000]
ctr["SEGMENT2"] = np.select(cond, ["Critique","Faible","Moyen","Élevé"], default="N/A")

# 3. LIB_STATUT
lib_dict = {1:"Ouvert",2:"En attente",3:"Suspendu",4:"Clôturé"}
ctr["LIB_STATUT2"] = ctr["COD_ECV_CTR"].map(lib_dict)

# 4. Date
ctr["DAT_OUV_CTR2"] = pd.to_datetime(ctr["DAT_OUV_CTR"], errors="coerce")
ctr["ANNEE_OUV2"]   = ctr["DAT_OUV_CTR2"].dt.year

print(ctr["ALERTE"].value_counts())
print(ctr["SEGMENT2"].value_counts())

## Module 6 — groupby() et agg()
### 6.1 Agrégation de base

In [ ]:
# ── groupby().agg() — GROUP BY en Pandas ─────────────────────────────────
# SELECT COD_ECV_CTR, COUNT(*) AS N, AVG(SLD_CTR) AS MOY FROM CTR GROUP BY ...

rapport = ctr.groupby("COD_ECV_CTR").agg(
    N     = ("IDT_AC",  "count"),    # nombre de lignes par groupe
    MOY   = ("SLD_CTR", "mean"),     # moyenne du solde
    TOTAL = ("SLD_CTR", "sum"),      # somme des soldes
    MIN   = ("SLD_CTR", "min"),      # solde minimum
    MAX   = ("SLD_CTR", "max")       # solde maximum
).reset_index()   # remettre COD_ECV_CTR comme colonne ordinaire

print(rapport.round(2))

In [ ]:
# ── Grouper sur plusieurs colonnes ───────────────────────────────────────
# Analyse par statut ET devise
par_statut_devise = ctr.groupby(["COD_ECV_CTR", "COD_DEV"]).agg(
    N      = ("IDT_AC", "count"),
    MOY    = ("SLD_CTR", "mean"),
    TOTAL  = ("SLD_CTR", "sum")
).reset_index()
print(par_statut_devise.round(2))

# HAVING equivalent — filtrer après l'agrégation
gros_groupes = par_statut_devise[par_statut_devise["N"] > 10]
print(gros_groupes)

### ✏️ Exercice 5 — Rapport transactions

In [ ]:
# ── EXERCICE 4 ─────────────────────────────────────────────────────────────
# Sur TXN_X_CTR :
# 1. groupby("IDT_AC") avec agg : N txn, MNT_TOTAL, MNT_MOYEN, MNT_MAX
# 2. Filtrer : comptes avec plus de 10 transactions
# 3. Trier par MNT_TOTAL décroissant
# 4. Afficher les 10 premiers

# --- votre code ---

### ✅ Correction Exercice 5

In [ ]:
# ── CORRECTION EXERCICE 4 ─────────────────────────────────────────────────
rapport_txn = txn.groupby("IDT_AC").agg(
    N_TXN    = ("NUM_ORD_MVT_CPB", "count"),
    MNT_TOT  = ("MNT_MVT", "sum"),
    MNT_MOY  = ("MNT_MVT", "mean"),
    MNT_MAX  = ("MNT_MVT", "max")
).reset_index()

# HAVING : garder seulement les comptes avec > 10 transactions
actifs = rapport_txn[rapport_txn["N_TXN"] > 10]

# Trier
top10 = actifs.sort_values("MNT_TOT", ascending=False).head(10)
print(top10.round(2))

## Module 7 — pd.merge() — Jointures
### 7.1 Jointure simple

In [ ]:
# ── pd.merge() — LEFT JOIN en Pandas ─────────────────────────────────────

# Joindre CTR avec TIE_X_CTR pour avoir IDT_PI sur chaque contrat
ctr_client = pd.merge(
    ctr,           # table gauche (TOUTES ses lignes gardées avec how="left")
    txc,           # table droite (TIE_X_CTR — lien contrat/client)
    on="IDT_AC",   # colonne de jointure commune aux deux tables
    how="left"     # LEFT JOIN = garder tous les contrats
)
print(f"Avant : {ctr.shape} | Après merge : {ctr_client.shape}")
print(ctr_client[["IDT_AC","IDT_PI","SLD_CTR"]].head(5))

In [ ]:
# ── Jointures chaînées — vue complète client + contrat + adresse ──────────

# Étape 1 : CTR + TIE_X_CTR (pour avoir IDT_PI)
step1 = pd.merge(ctr, txc, on="IDT_AC", how="left")

# Étape 2 : + TIE (infos client : type, langue, naissance...)
step2 = pd.merge(step1, tie, on="IDT_PI", how="left")

# Étape 3 : + TIE_ADR (adresse)
step3 = pd.merge(step2, tie_adr, on="IDT_PI", how="left")

print(f"Vue complète : {step3.shape}")
print(f"Colonnes : {list(step3.columns)}")

### ✏️ Exercice 6 — Jointure complète

In [ ]:
# ── EXERCICE 5 ─────────────────────────────────────────────────────────────
# Construire la vue complète CTR + TIE_X_CTR + TIE
# Puis :
# 1. Filtrer les particuliers (COD_TYP_TIE = 1) — PP
# 2. groupby("COD_LNG_CTR") → N, SLD_CTR moyen
# 3. Afficher le résultat

# --- votre code ---

### ✅ Correction Exercice 6

In [ ]:
# ── CORRECTION EXERCICE 5 ─────────────────────────────────────────────────
# Jointures
vue = pd.merge(pd.merge(ctr, txc, on="IDT_AC", how="left"),
               tie,  on="IDT_PI",  how="left")

# Filtrer PP
pp = vue[vue["COD_TYP_TIE"] == 1]
print(f"Contrats PP : {len(pp)}")

# Agréger par langue
resume_lng = pp.groupby("COD_LNG_CTR").agg(
    N   = ("IDT_AC", "count"),
    MOY = ("SLD_CTR", "mean")
).reset_index()
print(resume_lng.round(2))

## Module 8 — Nettoyage et synthèse : value_counts, isna, pivot_table, apply
### 8.1 value_counts et valeurs manquantes

In [ ]:
# ── value_counts() — fréquence de chaque valeur en une ligne ─────────────
repartition_statuts = ctr["COD_ECV_CTR"].value_counts()             # triée par fréquence décroissante
print(repartition_statuts)

repartition_pct = ctr["COD_ECV_CTR"].value_counts(normalize=True) * 100   # normalize=True → proportions
print(repartition_pct.round(1))

In [ ]:
# ── Valeurs manquantes : isna(), fillna(), dropna() ───────────────────────
# isna() renvoie True/False pour chaque cellule vide (NaN)
nb_manquants = ctr["SLD_CTR"].isna().sum()          # .sum() sur des booléens compte les True

print(f"Soldes manquants : {nb_manquants}")

# fillna(valeur) — remplacer les manquants par une valeur par défaut
ctr["SLD_CTR_COMPLET"] = ctr["SLD_CTR"].fillna(0)   # remplace chaque NaN par 0

# dropna(subset=[...]) — supprimer les lignes qui ont un manquant dans ces colonnes
ctr_sans_manquant = ctr.dropna(subset=["SLD_CTR"])
print(f"Lignes conservées après dropna : {len(ctr_sans_manquant)} / {len(ctr)}")

### 8.2 pivot_table — tableau croisé

In [ ]:
# ── pivot_table() — tableau croisé (équivalent d'un TCD Excel) ────────────
tableau_croise = ctr.pivot_table(
    values="SLD_CTR",              # colonne à agréger
    index="COD_ECV_CTR",           # une ligne du tableau par statut
    aggfunc=["mean", "count"]      # une ou plusieurs fonctions d'agrégation
)
print(tableau_croise.round(2))

### 8.3 apply — fonction personnalisée

In [ ]:
# ── apply() — appliquer une fonction personnalisée à chaque valeur ───────
def categoriser_solde(solde):
    if solde < 0:
        return "Négatif"
    elif solde < 5000:
        return "Faible"
    else:
        return "Confortable"

# apply() passe chaque valeur de la colonne à la fonction, une par une
ctr["CATEGORIE_SOLDE"] = ctr["SLD_CTR"].apply(categoriser_solde)
print(ctr[["IDT_AC", "SLD_CTR", "CATEGORIE_SOLDE"]].head())

### ✏️ Exercice 7 — Nettoyer et synthétiser TIE

In [ ]:
# ── EXERCICE 6 ─────────────────────────────────────────────────────────────
# TODO 1 : compter les valeurs manquantes de chaque colonne de `tie` avec .isna().sum()
# TODO 2 : remplacer les manquants de COD_LNG_CTR par "FR" (fillna)
# TODO 3 : joindre ctr, txc (on="IDT_AC") puis tie (on="IDT_PI"), puis construire
#          un pivot_table : moyenne de SLD_CTR par COD_LNG_CTR
# TODO 4 : ajouter une colonne CATEGORIE_SOLDE via apply() + categoriser_solde

### ✅ Correction Exercice 7

In [ ]:
# ── CORRECTION EXERCICE 6 ─────────────────────────────────────────────────
# 1. Valeurs manquantes par colonne
print(tie.isna().sum())

# 2. Compléter les manquants
tie["COD_LNG_CTR"] = tie["COD_LNG_CTR"].fillna("FR")

# 3. Jointure (ctr → txc sur IDT_AC, puis → tie sur IDT_PI) puis tableau croisé
jointure = pd.merge(ctr, txc, on="IDT_AC", how="left")
jointure = pd.merge(jointure, tie, on="IDT_PI", how="left")
tcd = jointure.pivot_table(values="SLD_CTR", index="COD_LNG_CTR", aggfunc="mean")
print(tcd.round(2))

# 4. Catégorisation
ctr["CATEGORIE_SOLDE"] = ctr["SLD_CTR"].apply(categoriser_solde)
print(ctr["CATEGORIE_SOLDE"].value_counts())

## Module 9 — Matplotlib : premiers graphiques
### 9.1 Graphique en barres

In [ ]:
# ── Matplotlib — structure de base d'un graphique ─────────────────────────
import matplotlib.pyplot as plt   # bibliothèque de graphiques

# Nombre de contrats par statut (déjà calculé avec value_counts au Module 7)
repartition = ctr["COD_ECV_CTR"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))                 # 1. créer la figure et les axes
ax.bar(repartition.index.astype(str), repartition.values, color="#0E8C86")  # 2. dessiner
ax.set_title("Nombre de contrats par statut")           # 3. titre
ax.set_xlabel("Code statut")
ax.set_ylabel("Nombre de contrats")
plt.show()                                               # 4. afficher

### 9.2 Histogramme — distribution des soldes

In [ ]:
# ── Histogramme — voir comment se répartissent des valeurs numériques ────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ctr["SLD_CTR"].dropna(), bins=15, color="#0B2D48", edgecolor="white")
ax.set_title("Distribution des soldes des contrats")
ax.set_xlabel("Solde (EUR)")
ax.set_ylabel("Nombre de contrats")
plt.show()

### 9.3 Enregistrer un graphique dans un fichier

In [ ]:
# ── Sauvegarder un graphique en image PNG ─────────────────────────────────
import os
os.makedirs("../output", exist_ok=True)   # créer le dossier de sortie s'il n'existe pas

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(repartition.index.astype(str), repartition.values, color="#0E8C86")
ax.set_title("Nombre de contrats par statut")
plt.savefig("../output/contrats_par_statut.png", dpi=150)   # enregistrer AVANT show()
plt.show()
print("Graphique enregistré : ../output/contrats_par_statut.png")

# Le Jour 3 approfondit Matplotlib : plus de types de graphiques, dashboard complet.

### ✏️ Exercice 8 — Graphique des devises

In [ ]:
# ── EXERCICE ────────────────────────────────────────────────────────────────
# TODO 1 : compter le nombre de contrats par devise (colonne COD_DEV) avec value_counts()
# TODO 2 : créer un graphique en barres HORIZONTALES (ax.barh)
# TODO 3 : ajouter un titre "Contrats par devise"
# TODO 4 : sauvegarder en PNG dans ../output/contrats_par_devise.png

### ✅ Correction Exercice 8

In [ ]:
# ── CORRECTION ─────────────────────────────────────────────────────────────
# 1. Comptage par devise
par_devise = ctr["COD_DEV"].value_counts()

# 2-3. Graphique en barres horizontales
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(par_devise.index, par_devise.values, color="#0E8C86")
ax.set_title("Contrats par devise")
ax.set_xlabel("Nombre de contrats")

# 4. Sauvegarde
plt.savefig("../output/contrats_par_devise.png", dpi=150)
plt.show()

## Module 10 — Boîte à outils Pandas : les fonctions essentielles
### 10.1 Créer un DataFrame à la main

In [ ]:
# ── Créer un DataFrame directement (sans lire un CSV) ─────────────────────
# Utile pour un test, un petit jeu de données, ou un résultat qu'on construit soi-même
mini_comptes = pd.DataFrame({
    "IDT_AC": ["AC001", "AC002", "AC003"],   # une clé = une colonne
    "SOLDE":  [1500.0, -200.0, 8300.0],
    "DEVISE": ["EUR", "EUR", "USD"],
})
print(mini_comptes)
print(type(mini_comptes))   # <class 'pandas.core.frame.DataFrame'>

### 10.2 unique() et nunique()

In [ ]:
# ── unique() et nunique() — valeurs distinctes d'une colonne ─────────────
print(ctr["COD_DEV"].unique())     # array des valeurs distinctes (ordre d'apparition)
print(ctr["COD_DEV"].nunique())    # nombre de valeurs distinctes (un seul chiffre)

### 10.3 duplicated() et drop_duplicates()

In [ ]:
# ── duplicated() et drop_duplicates() — détecter et retirer les doublons ──
# Un client (IDT_PI) peut apparaître plusieurs fois dans txc (plusieurs contrats)
print(f"Lignes dans txc          : {len(txc)}")
print(f"Clients (IDT_PI) uniques : {txc['IDT_PI'].nunique()}")

doublons = txc["IDT_PI"].duplicated()      # True pour chaque répétition (2ᵉ occurrence et plus)
print(f"Occurrences en double    : {doublons.sum()}")

txc_1_par_client = txc.drop_duplicates(subset="IDT_PI")   # garde la 1ère ligne par client
print(f"Après drop_duplicates    : {len(txc_1_par_client)}")

### 10.4 Méthodes de texte — l'accesseur .str

In [ ]:
# ── .str — méthodes de texte appliquées à TOUTE une colonne, sans boucle ──
# Même logique que les méthodes de chaînes du Jour 1 (.upper(), .strip()...),
# mais appliquées d'un coup à toute une colonne grâce à l'accesseur .str

libelles = txn["LIB_OPE_INL_1"].dropna()      # ignorer les libellés manquants

print(libelles.str.upper().head(3))                       # tout en majuscules
print(libelles.str.contains("SEPA").sum())                 # nb de libellés contenant "SEPA"
print(libelles.str.len().describe().round(1))               # longueur de chaque libellé
print(libelles.str.replace("BEOBANK", "BANQUE").head(3))   # remplacer un mot

### 10.5 assign() — ajouter des colonnes en chaîne

In [ ]:
# ── assign() — équivalent de ctr["COL"] = ..., mais s'enchaîne dans un pipeline ──
ctr_enrichi = ctr.assign(
    EST_NEGATIF = ctr["SLD_CTR"] < 0,             # nouvelle colonne 1
    SOLDE_ARRONDI = ctr["SLD_CTR"].round(0)       # nouvelle colonne 2
)
print(ctr_enrichi[["IDT_AC", "SLD_CTR", "EST_NEGATIF", "SOLDE_ARRONDI"]].head(3))

### 10.6 query() — filtrer avec une expression texte

In [ ]:
# ── query() — parfois plus lisible qu'un filtre par crochets ─────────────
# Équivalent de : ctr[ctr["COD_DEV"] == "EUR"]
en_euros = ctr.query("COD_DEV == 'EUR'")
print(f"Contrats en EUR : {len(en_euros)}")

# Plusieurs conditions dans la même expression (and / or acceptés directement)
gros_soldes_eur = ctr.query("COD_DEV == 'EUR' and SLD_CTR > 5000")
print(f"Gros soldes EUR : {len(gros_soldes_eur)}")

### 10.7 concat() — empiler des DataFrames

In [ ]:
# ── pd.concat() — empiler plusieurs DataFrames (ajouter des lignes) ──────
comptes_janvier = pd.DataFrame({"IDT_AC": ["AC001", "AC002"], "SOLDE": [1000, 2000]})
comptes_fevrier = pd.DataFrame({"IDT_AC": ["AC003"],          "SOLDE": [1500]})

tous_les_comptes = pd.concat([comptes_janvier, comptes_fevrier], ignore_index=True)
print(tous_les_comptes)

### 10.8 crosstab() — tableau croisé à deux entrées

In [ ]:
# ── pd.crosstab() — comme PROC FREQ avec 2 variables (TABLES A*B) ────────
tableau = pd.crosstab(ctr["COD_ECV_CTR"], ctr["COD_DEV"])
print(tableau)

### 10.9 .at / .iat — accès rapide à une seule valeur

In [ ]:
# ── .at (par label) et .iat (par position) — plus rapides que .loc/.iloc ──
# À utiliser quand on veut UNE seule valeur précise, pas une sélection
valeur_par_label    = ctr.at[0, "SLD_CTR"]     # ligne d'index 0, colonne "SLD_CTR"
valeur_par_position = ctr.iat[0, 7]            # ligne 0, 8ᵉ colonne (index 7, commence à 0)
print(valeur_par_label, valeur_par_position)

### 10.10 .copy() — éviter de modifier un DataFrame par accident

In [ ]:
# ── .copy() — obtenir une VRAIE copie indépendante ───────────────────────
# Sans .copy(), un sous-ensemble reste "lié" à l'original : le modifier peut
# déclencher un avertissement (SettingWithCopyWarning) ou modifier l'original par surprise.
sous_ensemble = ctr[ctr["COD_DEV"] == "EUR"].copy()   # .copy() = copie sûre et indépendante
sous_ensemble["NOTE"] = "vérifié"                       # modification sans risque

print("NOTE" in ctr.columns)   # False — ctr n'a pas été modifié

### ✏️ Exercice 9 — Boîte à outils

In [ ]:
# ── EXERCICE ────────────────────────────────────────────────────────────────
# TODO 1 : combien de devises distinctes dans ctr ? (nunique)
# TODO 2 : y a-t-il des IDT_AC en double dans ctr ? (duplicated().sum())
# TODO 3 : avec query(), filtrer les contrats en EUR avec un solde > 1000
# TODO 4 : construire un crosstab COD_ECV_CTR x COD_DEV

### ✅ Correction Exercice 9

In [ ]:
# ── CORRECTION ─────────────────────────────────────────────────────────────
# 1. Devises distinctes
print(f"1. Devises distinctes : {ctr['COD_DEV'].nunique()}")

# 2. Doublons sur IDT_AC
print(f"2. IDT_AC en double : {ctr['IDT_AC'].duplicated().sum()}")

# 3. query()
filtre = ctr.query("COD_DEV == 'EUR' and SLD_CTR > 1000")
print(f"3. Contrats EUR >1000 : {len(filtre)}")

# 4. crosstab
print("4. Statut x Devise :")
print(pd.crosstab(ctr["COD_ECV_CTR"], ctr["COD_DEV"]))

### 📎 Aide-mémoire — fonctions Pandas vues pendant la formation

| Catégorie | Fonctions |
|---|---|
| Lecture / écriture | `pd.read_csv()`, `.to_csv()`, `.to_sql()`, `pd.read_sql()` |
| Explorer | `.head()`, `.tail()`, `.shape`, `.info()`, `.dtypes`, `.describe()`, `.columns` |
| Valeurs distinctes | `.unique()`, `.nunique()`, `.value_counts()` |
| Doublons | `.duplicated()`, `.drop_duplicates()` |
| Sélection | `.loc[]`, `.iloc[]`, `.at[]`, `.iat[]` |
| Filtrer | `df[condition]`, `.isin()`, `.between()`, `.isna()`, `.notna()`, `.query()` |
| Trier | `.sort_values()`, `.sort_index()` |
| Colonnes | `.assign()`, `.drop()`, `.rename()`, `.copy()` |
| Colonnes calculées | `np.where()`, `np.select()`, `.map()`, `.apply()`, `pd.cut()` |
| Texte | accesseur `.str` (`.str.upper()`, `.str.contains()`, `.str.replace()`, `.str.len()`...) |
| Dates | `pd.to_datetime()`, accesseur `.dt` (`.dt.year`, `.dt.month`...) |
| Manquants | `.isna()`, `.fillna()`, `.dropna()` |
| Agrégation | `.groupby().agg()`, `.pivot_table()`, `pd.crosstab()` |
| Combiner | `pd.merge()`, `pd.concat()` |
| Graphiques | `.plot()` (via Matplotlib, voir Module Matplotlib et Jour 3) |

## Module 11 — Export et SQLite
### 11.1 to_csv et SQLite

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────────

# Créer le dossier output s'il n'existe pas
import os
os.makedirs("../output", exist_ok=True)

# Export CSV (même convention que les fichiers Beobank source)
rapport.to_csv(
    "../output/rapport_statuts.csv",
    sep=";",           # séparateur point-virgule
    index=False,       # ne PAS écrire la colonne d'index numérique
    encoding="utf-8"
)
print("Export CSV terminé.")

# ── SQLite — requêtes SQL sur les données Pandas ──────────────────────────
conn = sqlite3.connect(":memory:")      # base de données en mémoire RAM

# Charger les DataFrames dans SQLite
ctr.to_sql("CTR",        conn, if_exists="replace", index=False)
txn.to_sql("TXN_X_CTR",  conn, if_exists="replace", index=False)
txc.to_sql("TIE_X_CTR",  conn, if_exists="replace", index=False)
tie.to_sql("TIE",        conn, if_exists="replace", index=False)

# Requête SQL équivalente
sql = """
    SELECT c.COD_ECV_CTR,
           COUNT(*)         AS N,
           AVG(c.SLD_CTR)   AS MOY_SOLDE,
           SUM(t.MNT_MVT)   AS TOTAL_TXN
    FROM CTR c
    LEFT JOIN TXN_X_CTR t ON c.IDT_AC = t.IDT_AC
    GROUP BY c.COD_ECV_CTR
    ORDER BY N DESC
"""
resultat_sql = pd.read_sql(sql, conn)
print(resultat_sql.round(2))

## Exercice Final — Rapport mensuel PP (pipeline complet)

In [ ]:
# ── EXERCICE FINAL ─────────────────────────────────────────────────────────
# Pipeline : CTR + TIE_X_CTR + TIE → filtrer PP → enrichir → agréger → exporter
#
# Étapes :
# 1. Jointure CTR + TIE_X_CTR + TIE
# 2. Filtrer COD_TYP_TIE = 1 (PP)
# 3. Créer LIB_STATUT avec .map()
# 4. Créer SEGMENT avec np.select
# 5. groupby("COD_LNG_CTR") → N, SLD_CTR moyen, SLD_CTR total
# 6. Exporter résultat en CSV

# --- votre code ---

### ✅ Correction Exercice Final

In [ ]:
# ── CORRECTION EXERCICE FINAL ─────────────────────────────────────────────
# 1-2. Jointure + filtre PP
vue_pp = pd.merge(pd.merge(ctr, txc, on="IDT_AC", how="left"),
                  tie, on="IDT_PI", how="left")
pp = vue_pp[vue_pp["COD_TYP_TIE"] == 1].copy()

# 3. Libellés statuts
lib_dict = {1:"Ouvert",2:"En attente",3:"Suspendu",
            4:"Clôturé",5:"En résiliation",6:"Résilié"}
pp["LIB_STATUT"] = pp["COD_ECV_CTR"].map(lib_dict)

# 4. Segment
cond = [pp["SLD_CTR"]<0, pp["SLD_CTR"]<5000,
        pp["SLD_CTR"]<50000, pp["SLD_CTR"]>=50000]
pp["SEGMENT"] = np.select(cond, ["Critique","Faible","Moyen","Élevé"], default="N/A")

# 5. Agrégation par langue
rapport_pp = pp.groupby("COD_LNG_CTR").agg(
    N      = ("IDT_AC", "count"),
    MOY    = ("SLD_CTR", "mean"),
    TOTAL  = ("SLD_CTR", "sum")
).reset_index()
print("=== Rapport mensuel PP par langue ===")
print(rapport_pp.round(2))

# 6. Export
import os; os.makedirs("../output", exist_ok=True)
rapport_pp.to_csv("../output/rapport_mensuel_pp.csv", sep=";", index=False)
print("\nExport terminé : ../output/rapport_mensuel_pp.csv")